In [10]:
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=True)
    page = browser.new_page()
    
    # Navegar a la página del torneo
    page.goto("https://www.sofascore.com/es/torneo/futbol/mexico/liga-mx-apertura/11621")
    
    # Esperar que carguen los partidos de la ronda 10
    page.wait_for_selector("div[data-test='event-card']")

    # Extraer datos de la ronda 10
    events = page.evaluate("""
        () => {
            return Array.from(document.querySelectorAll("div[data-test='event-card']")).map(e => ({
                home: e.querySelector(".home").innerText,
                away: e.querySelector(".away").innerText,
                score: e.querySelector(".score").innerText,
                date: e.querySelector(".date").innerText
            }));
        }
    """)
    print(events)
    browser.close()


Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

In [13]:
import json
import time
from datetime import datetime

# Cargar el JSON desde un archivo
with open("liga_mx_apertura.json", "r", encoding="utf-8") as f:
    data = json.load(f)

ahora = time.time()

for round_number, round_data in sorted(data.items(), key=lambda x: int(x[0])):
    events = round_data.get("events", [])
    print(f"Ronda {round_number}:")
    
    for event in events:
        # Extraer equipos
        home = event.get("homeTeam", {}).get("name", "Desconocido")
        away = event.get("awayTeam", {}).get("name", "Desconocido")
        
        # Extraer scores
        home_score = event.get("homeScore", {}).get("display")
        away_score = event.get("awayScore", {}).get("display")
        
        # Extraer timestamp y convertir a fecha legible
        ts = event.get("startTimestamp", 0)
        fecha = datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M")
        
        # Revisar si el partido es futuro
        proximo = ts > ahora
        
        # Formato de salida
        score_str = f"{home_score}-{away_score}" if home_score is not None and away_score is not None else "None-None"
        marcador = f"{home} vs {away} | {score_str} | {fecha}"
        
        # Marcar próximos partidos con *
        if proximo:
            marcador += "  *PRÓXIMO*"
        
        print(marcador)
    print("-" * 50)


Ronda 1:
Club Puebla vs Atlas FC | 2-3 | 2025-07-11 20:00
FC Juárez vs Club América | 1-1 | 2025-07-11 22:00
Club Tijuana vs Querétaro FC | 1-0 | 2025-07-11 22:05
CD Toluca vs Club Necaxa | 3-1 | 2025-07-12 20:00
Santos Laguna vs Pumas UNAM | 3-0 | 2025-07-12 20:00
Cruz Azul vs Mazatlán FC | 0-0 | 2025-07-12 22:05
CF Pachuca vs CF Monterrey | 3-0 | 2025-07-13 18:00
Club León vs Atlético San Luis | 0-1 | 2025-07-13 20:00
CD Guadalajara vs Tigres UANL | 0-0 | 2025-09-17 20:05
--------------------------------------------------
Ronda 2:
Club América vs Club Tijuana | 3-1 | 2025-07-16 20:00
Santos Laguna vs CD Toluca | 2-4 | 2025-07-16 22:05
Club Necaxa vs Querétaro FC | 3-1 | 2025-07-18 20:00
Atlético San Luis vs CF Monterrey | 0-1 | 2025-07-18 22:00
Mazatlán FC vs Club Puebla | 2-1 | 2025-07-18 22:05
Club León vs CD Guadalajara | 1-0 | 2025-07-19 20:00
Tigres UANL vs FC Juárez | 1-0 | 2025-07-19 20:00
Atlas FC vs Cruz Azul | 3-3 | 2025-07-19 22:05
Pumas UNAM vs CF Pachuca | 2-3 | 2025-07-

In [14]:
import json

# Carga el archivo JSON
with open("liga_mx_apertura_2025.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Usamos un set para evitar duplicados
equipos = set()

# Recorremos todas las jornadas y partidos
for round_number, round_data in data.items():
    for event in round_data.get("events", []):
        home = event.get("homeTeam", {}).get("name")
        away = event.get("awayTeam", {}).get("name")
        if home:
            equipos.add(home)
        if away:
            equipos.add(away)

# Convertimos a lista ordenada alfabéticamente
equipos = sorted(list(equipos))

# Mostramos la lista
for equipo in equipos:
    print(equipo)


Atlas FC
Atlético San Luis
CD Guadalajara
CD Toluca
CF Monterrey
CF Pachuca
Club América
Club León
Club Necaxa
Club Puebla
Club Tijuana
Cruz Azul
FC Juárez
Mazatlán FC
Pumas UNAM
Querétaro FC
Santos Laguna
Tigres UANL
